# Simulation-Based Side-Channel Attack on Mechanical Control Systems
### Academic Project — Google Colab Notebook

**Runs fully in the cloud. No hardware or MATLAB needed.**

| Phase | What happens |
|-------|--------------|
| Step 1 | Clone repo from GitHub |
| Step 2 | Generate 15,000 synthetic power traces |
| Step 3 | Run SPA — visualise AES round structure |
| Step 4 | Run DPA — recover AES-128 key |
| Step 5 | Run CPA — more efficient key recovery + rho values |
| Step 6 | Classify motor load state from power traces |
| Step 7 | Simulate countermeasures (jitter, masking, noise) |
| Step 8 | Generate all publication figures |

## Step 1 — Clone Your GitHub Repository

In [ ]:
# ── Replace with YOUR GitHub username and repo name ──────────────
GITHUB_USERNAME = "YOUR_USERNAME"
REPO_NAME       = "sca-mechanical-systems"
# ─────────────────────────────────────────────────────────────────

import os

repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
!git clone {repo_url}

# Move into the src folder so all imports resolve
os.chdir(f"/content/{REPO_NAME}/src")
print("Working directory:", os.getcwd())

# Create output directories
for d in ["../traces", "../results", "../plots"]:
    os.makedirs(d, exist_ok=True)
print("Directories ready.")

## Step 2 — Install Dependencies & Generate Power Traces

In [ ]:
!pip install -q numpy scipy matplotlib pandas scikit-learn
print("Libraries ready.")

In [ ]:
# Generate all power traces (takes ~2 minutes)
# This creates 6 scenario files: 3 loads x 2 noise levels
# No MATLAB needed — uses built-in synthetic motor current model

from trace_generator import generate_traces

print("Generating power traces (synthetic DC motor + AES Hamming Weight model)")
print("-" * 60)

for load_id in [0, 1, 2]:
    for sigma in [0.05, 0.15]:
        generate_traces(load_id, sigma, n_traces=5000)

print("-" * 60)
print("Done. Trace files saved in ../traces/")

## Step 3 — Simple Power Analysis (SPA)

In [ ]:
# SPA: visualise the AES round structure in averaged traces
# You should see 10 distinct bumps — one per AES round

from spa_attack import run_spa

print("=" * 50)
print("Simple Power Analysis — No-load scenario")
print("=" * 50)
run_spa(load_id=0, sigma=0.05, n_avg=20)

## Step 4 — Differential Power Analysis (DPA)

In [ ]:
# DPA: statistically recover all 16 AES key bytes
# Red line in the plot = correct key hypothesis

from dpa_attack import recover_full_key

print("=" * 50)
print("DPA — No load, low noise (sigma=0.05)")
print("=" * 50)
c1 = recover_full_key(load_id=0, sigma=0.05)

print("\n" + "=" * 50)
print("DPA — Full load, high noise (sigma=0.15)")
print("=" * 50)
c2 = recover_full_key(load_id=2, sigma=0.15)

## Step 5 — Correlation Power Analysis (CPA)

In [ ]:
# CPA: more efficient than DPA, gives you the rho correlation values
# for Table I in the paper

from cpa_attack import cpa_full_key
import pandas as pd

print("Running CPA across all 6 scenarios...")
print("=" * 55)

rows = []
for load_id in [0, 1, 2]:
    for sigma in [0.05, 0.15]:
        correct, mean_rho = cpa_full_key(load_id, sigma)
        rows.append({
            "Scenario"  : f"L{load_id}, sigma={sigma}",
            "Correct"   : f"{correct}/16",
            "Mean rho"  : round(mean_rho, 3)
        })

df = pd.DataFrame(rows)
print("\n")
print(df.to_string(index=False))

import os
df.to_csv("../results/cpa_results.csv", index=False)
print("\nSaved to ../results/cpa_results.csv")

## Step 6 — Mechanical State Classifier
### Can we infer motor load from power traces WITHOUT breaking the cipher?

In [ ]:
# Novel finding: motor load state leaks through the power trace
# even without any knowledge of the AES key or plaintext

from mechanical_classifier import classify_load

print("=" * 50)
print("Load Classification — Low noise (sigma=0.05)")
print("=" * 50)
acc_low = classify_load(sigma=0.05)

print("\n" + "=" * 50)
print("Load Classification — High noise (sigma=0.15)")
print("=" * 50)
acc_high = classify_load(sigma=0.15)

print(f"\nSummary:")
print(f"  sigma=0.05 accuracy: {acc_low*100:.1f}%")
print(f"  sigma=0.15 accuracy: {acc_high*100:.1f}%")

## Step 7 — Countermeasures

In [ ]:
# Compare 4 defences: timing jitter, noise injection,
# Boolean masking, and masking+jitter combined

from countermeasures import compare_countermeasures

print("=" * 50)
print("Countermeasure Effectiveness (L0, sigma=0.05)")
print("=" * 50)
rhos = compare_countermeasures(load_id=0, sigma=0.05)

print("\nTable II values:")
for name, rho in rhos.items():
    broken = "Yes" if rho > 0.2 else "No"
    print(f"  {name:20s}  rho={rho:.2f}  Broken={broken}")

## Step 8 — Generate Publication Figures

In [ ]:
from plot_results import fig1_trace_comparison, fig2_rho_bar, fig3_countermeasures

print("Generating figures for paper...")
fig1_trace_comparison()
fig2_rho_bar()
fig3_countermeasures()
print("All figures saved to ../plots/")

## Step 9 — Download All Results

In [ ]:
# Zip plots + results and download them
import shutil
from google.colab import files

shutil.make_archive("/content/sca_results", "zip", "/content",
                    base_dir=None,
                    root_dir=f"/content/{REPO_NAME}",
                    base_dir="plots")

shutil.make_archive("/content/sca_results", "zip",
                    f"/content/{REPO_NAME}")

# Download plots folder
shutil.make_archive("/content/plots_only", "zip",
                    root_dir=f"/content/{REPO_NAME}",
                    base_dir="plots")

files.download("/content/plots_only.zip")
print("Download started — check your browser downloads folder.")